imports

In [1]:
import pandas as pd 
import numpy as np
import cv2 
import os
import chromadb
from insightface.app import FaceAnalysis

/home/arc/tensuur/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider'])
model.prepare(ctx_id=0, det_size=(640, 640))

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/arc/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], wi

In [3]:
chroma_client = chromadb.PersistentClient(path = "sdf")
collection = chroma_client.get_or_create_collection(name="face_embeddings",
                                                    metadata={"hnsw:space": "cosine"})

In [4]:
import uuid
import shutil

### Quality filter

In [5]:
def quality(img,face,blur,conf,ratio):
    if face.det_score < conf:
        return False
    bbox = face.bbox.astype(int)
    x1, y1, x2, y2 = bbox
    y1,y2 = max(0,y1), min(img.shape[0],y2)
    x1,x2 = max(0,x1), min(img.shape[1],x2)
    face_crop = img[y1:y2, x1:x2]
    if face_crop.size == 0:
        return False
    
    img_area = img.shape[0] * img.shape[1]
    face_area = (x2 - x1) * (y2 - y1)
    ratios = face_area / img_area
    if ratios < ratio:
        return False
    
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var < blur:
        return False
    return True

same dirs as before

In [6]:
dump = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb"
output = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/facial_output"
os.makedirs(output, exist_ok=True)
skipp = os.path.join(output,"skipped")
os.makedirs(skipp,exist_ok=True)

In [7]:
blur  = 80
conf = 0.8
ratio = 0.04

### calculating the number of embedding after applying filter

In [12]:
all_embed = []
for folder in os.listdir(dump):
    for img_name in os.listdir(os.path.join(dump,folder)):
        img_path = os.path.join(os.path.join(dump,folder),img_name)
        img = cv2.imread(img_path)
        faces = model.get(img)
        for face in faces:
            blur_check = quality(img,face,blur,conf,ratio)
            if not blur_check:
                continue
            all_embed.append({"path":img_path,"embed":face.embedding,"name": img_name})


/home/arc/tensuur/lib/python3.13/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


In [13]:
print(len(all_embed))

66


In [14]:
num = 5
head = 2

### Clustering

In [18]:
from sklearn.cluster import AgglomerativeClustering
embed_mat = np.array([f["embed"] for f in all_embed])
target = num+head
cluster_algo = AgglomerativeClustering(n_clusters=target,metric="cosine",linkage="average")
labels = cluster_algo.fit_predict(embed_mat)
for i,face_data in enumerate(all_embed):
    face_data["id"] = labels[i]

In [19]:
print(all_embed)

[{'path': '/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb/madonna/httpimagegaladevcmseamadonnaprivatdetektivsquaretopsquarejpgv.jpg', 'embed': array([-8.87394324e-02, -2.74824888e-01, -6.64048433e-01, -5.83479464e-01,
       -1.13655579e+00, -6.24587476e-01, -8.46399128e-01, -6.90623760e-01,
       -1.53597236e+00,  1.79216278e+00,  3.47691596e-01, -4.23433006e-01,
       -6.95968091e-01, -1.64795732e+00,  1.22587085e-01, -7.71171808e-01,
       -9.50842738e-01, -8.37807804e-02, -5.86204410e-01,  4.54331413e-02,
        1.08572257e+00, -6.99820995e-01,  1.82261896e+00,  7.03611612e-01,
        7.16060519e-01,  4.75709289e-01,  9.37733114e-01,  5.89643717e-02,
        3.21149528e-01,  2.03766918e+00,  1.27076066e+00, -4.00412530e-01,
       -3.13416481e-01, -1.01866317e+00, -7.34256268e-01,  1.95705831e-01,
       -8.40398669e-01,  4.80875909e-01,  3.46214861e-01,  6.95178360e-02,
       -5.01798153e-01,  1.08611619e+00, -3.27942252e-01,  2.10909694e-01,
  

In [ ]:
import shutil
for face in all_embed:
    target_dir = os.path.join(output,face["id"])
    os.makedirs(target_dir,exist_ok=True)
    target_path = os.path.join(target_dir,face["name"])
    if not os.path.exists(target_path):
        shutil.copy()